In [1]:
from google.colab import files
uploaded = files.upload()

Saving tomato_leaves_classification.zip to tomato_leaves_classification.zip


In [2]:
import zipfile
zipfile.ZipFile('tomato_leaves_classification.zip').extractall('data')

In [3]:
import os

base_dir = '/content/data/tomato_leaves_classification'
print(os.listdir(base_dir))

['Tomato___Late_blight', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___healthy', 'Tomato___Bacterial_spot', 'Tomato___Target_Spot', 'Tomato___Early_blight', 'Tomato___Tomato_mosaic_virus']


In [4]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import numpy as np
import os

In [5]:
data_dir = "/content/data/tomato_leaves_classification"

img_size = (224, 224)
batch_size = 32

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

class_names = train_ds.class_names
print("Classes:", class_names)

Found 18160 files belonging to 10 classes.
Using 14528 files for training.
Found 18160 files belonging to 10 classes.
Using 3632 files for validation.
Classes: ['Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus', 'Tomato___healthy']


In [6]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

In [7]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=img_size + (3,),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

inputs = keras.Input(shape=img_size + (3,))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)

model = keras.Model(inputs, outputs)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [8]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [9]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

Epoch 1/5
454/454 ━━━━━━━━━━━━━━━━━━━━ 753s 2s/step - accuracy: 0.7734 - loss: 0.6999 - val_accuracy: 0.8739 - val_loss: 0.4069
Epoch 2/5
454/454 ━━━━━━━━━━━━━━━━━━━━ 680s 1s/step - accuracy: 0.8815 - loss: 0.3678 - val_accuracy: 0.8932 - val_loss: 0.3340
Epoch 3/5
454/454 ━━━━━━━━━━━━━━━━━━━━ 686s 2s/step - accuracy: 0.8986 - loss: 0.3075 - val_accuracy: 0.9064 - val_loss: 0.2963
Epoch 4/5
454/454 ━━━━━━━━━━━━━━━━━━━━ 682s 2s/step - accuracy: 0.9099 - loss: 0.2684 - val_accuracy: 0.9058 - val_loss: 0.2872
Epoch 5/5
454/454 ━━━━━━━━━━━━━━━━━━━━ 683s 2s/step - accuracy: 0.9193 - loss: 0.2466 - val_accuracy: 0.9122 - val_loss: 0.2626


In [20]:
model.save("tomato_model.keras")

In [21]:
from google.colab import files
files.download("tomato_model.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>